# Лабораторная работа №2: Фильтрация изображений, шумоподавление и математическая морфология
**Цель работы:**
1. Исследование и моделирование различных типов шума (гауссов шум, импульсный шум «соль-перец»).
2. Применение линейных сглаживающих фильтров (Box filter, фильтр Гаусса).
3. Применение нелинейных и адаптивных фильтров (медианный, билатеральный фильтры).
4. Подавление шумов методами математической морфологии (эрозия, дилатация, открытие, закрытие).
5. Оценка качества фильтрации метриками MSE и PSNR.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 6)

## 1. Загрузка исходных изображений
Загрузим монохромный радиолокационный снимок `sar_1.jpg` и цветную фотографию `img.jpg`.

In [ ]:
# Загрузка SAR-снимка
img_sar = cv2.imread('sar_1.jpg')
if img_sar is not None:
    img_sar_gray = cv2.cvtColor(img_sar, cv2.COLOR_BGR2GRAY)
else:
    raise FileNotFoundError("Файл sar_1.jpg не найден!")

# Загрузка цветного снимка
img_color = cv2.imread('img.jpg')
if img_color is not None:
    img_color_rgb = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)
else:
    raise FileNotFoundError("Файл img.jpg не найден!")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_sar_gray, cmap='gray')
axes[0].set_title(f'SAR-снимок (sar_1.jpg) - {img_sar_gray.shape}')
axes[0].axis('off')

axes[1].imshow(img_color_rgb)
axes[1].set_title(f'Цветное изображение (img.jpg) - {img_color_rgb.shape}')
axes[1].axis('off')
plt.show()

## 2. Моделирование шума
Реализуем функции добавления гауссова шума и импульсного шума «соль-перец».

In [ ]:
# 1. Гауссов шум
def add_gaussian_noise(image, mean=0, stddev=30):
    noise = np.zeros(image.shape, dtype=np.uint8)
    cv2.randn(noise, mean, stddev)
    noisy_image = cv2.add(image, noise)
    return noisy_image, noise

# 2. Шум «соль-перец»
def add_salt_and_pepper_noise(image, prob=0.06):
    noisy = image.copy()
    rnd = np.random.rand(*image.shape[:2])
    noisy[rnd < (prob / 2)] = 0         # перец
    noisy[rnd > 1 - (prob / 2)] = 255   # соль
    return noisy

sar_gauss, noise_map = add_gaussian_noise(img_sar_gray, mean=0, stddev=40)
sar_sp = add_salt_and_pepper_noise(img_sar_gray, prob=0.08)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(img_sar_gray, cmap='gray')
axes[0].set_title('Оригинал')
axes[1].imshow(noise_map, cmap='gray')
axes[1].set_title('Карта гауссова шума')
axes[2].imshow(sar_gauss, cmap='gray')
axes[2].set_title('Гауссов шум (std=40)')
axes[3].imshow(sar_sp, cmap='gray')
axes[3].set_title('Шум "Соль-перец" (8%)')
for ax in axes: ax.axis('off')
plt.show()

## 3. Линейная и нелинейная фильтрация
Применим линейное сглаживание (Box Filter, Gaussian Blur) и ранговые/адаптивные фильтры (Median, Bilateral).

In [ ]:
ksize = 5
# Фильтрация гауссова шума
gauss_blur = cv2.blur(sar_gauss, (ksize, ksize))
gauss_filtered = cv2.GaussianBlur(sar_gauss, (ksize, ksize), sigmaX=1.5)
gauss_median = cv2.medianBlur(sar_gauss, ksize)
gauss_bilateral = cv2.bilateralFilter(sar_gauss, d=9, sigmaColor=75, sigmaSpace=75)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
axes[0].imshow(sar_gauss, cmap='gray')
axes[0].set_title('Зашумленное (Гаусс)')
axes[1].imshow(gauss_blur, cmap='gray')
axes[1].set_title(f'Box Filter {ksize}x{ksize}')
axes[2].imshow(gauss_filtered, cmap='gray')
axes[2].set_title('Gaussian Blur')
axes[3].imshow(gauss_median, cmap='gray')
axes[3].set_title('Median Filter')
axes[4].imshow(gauss_bilateral, cmap='gray')
axes[4].set_title('Bilateral Filter')
for ax in axes: ax.axis('off')
plt.suptitle('Фильтрация гауссова шума', fontsize=14)
plt.show()

In [ ]:
# Фильтрация шума «соль-перец»
sp_blur = cv2.blur(sar_sp, (ksize, ksize))
sp_gaussian = cv2.GaussianBlur(sar_sp, (ksize, ksize), sigmaX=1.5)
sp_median = cv2.medianBlur(sar_sp, ksize)
sp_bilateral = cv2.bilateralFilter(sar_sp, d=9, sigmaColor=75, sigmaSpace=75)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
axes[0].imshow(sar_sp, cmap='gray')
axes[0].set_title('Зашумленное (Соль-перец)')
axes[1].imshow(sp_blur, cmap='gray')
axes[1].set_title(f'Box Filter {ksize}x{ksize}')
axes[2].imshow(sp_gaussian, cmap='gray')
axes[2].set_title('Gaussian Blur')
axes[3].imshow(sp_median, cmap='gray')
axes[3].set_title('Median Filter (Лучший)')
axes[4].imshow(sp_bilateral, cmap='gray')
axes[4].set_title('Bilateral Filter')
for ax in axes: ax.axis('off')
plt.suptitle('Фильтрация импульсного шума "соль-перец"', fontsize=14)
plt.show()

## 4. Методы математической морфологии
Исследуем действие морфологических операций:
* Эрозия (Erosion) удаляет изолированные белые точки («соль»);
* Дилатация (Dilation) заполняет черные точки («перец»);
* Открытие (Opening) и Закрытие (Closing) возвращают объектам исходный размер.

In [ ]:
# Получение бинарного изображения и зашумление
_, sar_bin = cv2.threshold(img_sar_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
sar_bin_noisy = add_salt_and_pepper_noise(sar_bin, prob=0.06)

# Структурирующий элемент
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

erosion = cv2.erode(sar_bin_noisy, kernel, iterations=1)
dilation = cv2.dilate(sar_bin_noisy, kernel, iterations=1)
opening = cv2.morphologyEx(sar_bin_noisy, cv2.MORPH_OPEN, kernel)
closing = cv2.morphologyEx(sar_bin_noisy, cv2.MORPH_CLOSE, kernel)
open_close = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0, 0].imshow(sar_bin_noisy, cmap='gray')
axes[0, 0].set_title('Бинарное с шумом (Соль-перец)')
axes[0, 1].imshow(erosion, cmap='gray')
axes[0, 1].set_title('Эрозия (удаляет белые точки)')
axes[0, 2].imshow(dilation, cmap='gray')
axes[0, 2].set_title('Дилатация (закрывает черные точки)')
axes[1, 0].imshow(opening, cmap='gray')
axes[1, 0].set_title('Открытие (Opening)')
axes[1, 1].imshow(closing, cmap='gray')
axes[1, 1].set_title('Закрытие (Closing)')
axes[1, 2].imshow(open_close, cmap='gray')
axes[1, 2].set_title('Opening + Closing (Очищенное)')
for ax in axes.ravel(): ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Шумоподавление цветных изображений
Сравнение билинейного и алгоритма нелокальных средних (Non-Local Means) на цветном снимке `img.jpg`.

In [ ]:
color_noise = np.zeros(img_color_rgb.shape, dtype=np.uint8)
cv2.randn(color_noise, 0, 30)
color_noisy = cv2.add(img_color_rgb, color_noise)

denoised_nlm = cv2.fastNlMeansDenoisingColored(color_noisy, None, h=10, hColor=10, templateWindowSize=7, searchWindowSize=21)
denoised_bilateral = cv2.bilateralFilter(color_noisy, d=9, sigmaColor=75, sigmaSpace=75)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(color_noisy)
axes[0].set_title('Зашумленный цветок (Гаусс)')
axes[0].axis('off')

axes[1].imshow(denoised_bilateral)
axes[1].set_title('Bilateral Filter (RGB)')
axes[1].axis('off')

axes[2].imshow(denoised_nlm)
axes[2].set_title('Non-Local Means Denoising')
for ax in axes: ax.axis('off')
plt.show()

## 6. Количественная оценка качества фильтрации (MSE и PSNR)

In [ ]:
def calculate_metrics(original, processed):
    mse = np.mean((original.astype('float') - processed.astype('float')) ** 2)
    if mse == 0:
        return 0, float('inf')
    psnr = 10 * np.log10((255.0 ** 2) / mse)
    return mse, psnr

print("=== Оценка фильтрации шума 'Соль-перец' на SAR снимке ===")
filters = {
    'Зашумленное (Без фильтра)': sar_sp,
    'Box Filter': sp_blur,
    'Gaussian Filter': sp_gaussian,
    'Median Filter': sp_median,
    'Bilateral Filter': sp_bilateral
}

for name, filtered_img in filters.items():
    mse_val, psnr_val = calculate_metrics(img_sar_gray, filtered_img)
    print(f"{name:<26} | MSE: {mse_val:8.2f} | PSNR: {psnr_val:5.2f} dB")
